# MATH 5110 Lab 2a Study File  
## Chromaticity Conversion and Barycentric Coordinates

**Purpose.** This independent-study notebook explains how to convert a point $(x,y)$ on the chromaticity diagram into a unique triple $(r,b,g)$ of coefficients for red, blue, and green.

The core question is:

> How can three color vectors $R,B,G$ in the plane $\mathbb R^2$ represent a point $X\in\mathbb R^2$ uniquely?

Because three vectors in $\mathbb R^2$ are linearly dependent, the representation is not unique unless we add one extra condition:

$$
r+b+g=1.
$$

These are **barycentric coordinates** of $X$ with respect to the triangle formed by $R,B,G$.

## 1. Data from the lab

The primary color points are

$$
R=\begin{bmatrix}0.67\\0.33\end{bmatrix},\qquad
B=\begin{bmatrix}0.14\\0.08\end{bmatrix},\qquad
G=\begin{bmatrix}0.21\\0.71\end{bmatrix}.
$$

The order used in this notebook is

$$
(r,b,g)=\text{coefficients of }(R,B,G).
$$

Thus

$$
X=rR+bB+gG.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

R = np.array([0.67, 0.33])
B = np.array([0.14, 0.08])
G = np.array([0.21, 0.71])

print("R =", R)
print("B =", B)
print("G =", G)

## 2. Visualizing the RGB triangle

The triangle with vertices $R,B,G$ is the set of all mixtures

$$
rR+bB+gG,\qquad r,b,g\ge 0,\qquad r+b+g=1.
$$

If one coefficient is negative, the point lies outside the RGB triangle. In the physical chromaticity interpretation, negative coefficients mean that one primary must be moved to the other side of the matching equation.

In [ ]:
triangle = np.vstack([R, B, G, R])
W = (R+B+G)/3

plt.figure(figsize=(6,6))
plt.plot(triangle[:,0], triangle[:,1], marker="o")
for name, P in [("R", R), ("B", B), ("G", G)]:
    plt.text(P[0]+0.01, P[1]+0.01, name, fontsize=14)
plt.scatter([W[0]], [W[1]], s=80)
plt.text(W[0]+0.01, W[1]+0.01, "W", fontsize=14)
plt.xlabel("x")
plt.ylabel("y")
plt.title("RGB Triangle and White Point")
plt.grid(True)
plt.axis("equal")
plt.show()

print("White point W =", W)

## 3. Task 1: Find a linear dependence among $R,B,G$

Find nonzero numbers $u,v,w$ such that

$$
uR+vB+wG=0.
$$

This means $(u,v,w)^T$ is in the null space of

$$
M=\begin{bmatrix}R&B&G\end{bmatrix}
=
\begin{bmatrix}
0.67&0.14&0.21\\
0.33&0.08&0.71
\end{bmatrix}.
$$

In [ ]:
R_exact = sp.Matrix([sp.Rational(67,100), sp.Rational(33,100)])
B_exact = sp.Matrix([sp.Rational(14,100), sp.Rational(8,100)])
G_exact = sp.Matrix([sp.Rational(21,100), sp.Rational(71,100)])

M = sp.Matrix.hstack(R_exact, B_exact, G_exact)
null_vec = M.nullspace()[0]
u, v, w = null_vec
s = sp.simplify(u + v + w)

print("M =")
display(M)
print("A null vector is")
display(null_vec)
print("u =", u)
print("v =", v)
print("w =", w)
print("s = u+v+w =", s)
print("Check M*[u,v,w]^T =")
display(M*null_vec)

One convenient choice is

$$
u=\frac{413}{37},\qquad v=-\frac{2032}{37},\qquad w=1.
$$

Equivalently,

$$
413R-2032B+37G=0.
$$

Also

$$
s=u+v+w=-\frac{1582}{37}.
$$

## 4. Task 2: Find the matrix from $(x,y)$ to $(c,d)$

First represent $X$ using only $R$ and $B$:

$$
X=cR+dB.
$$

Let

$$
P=\begin{bmatrix}R&B\end{bmatrix}.
$$

Then

$$
X=P\begin{bmatrix}c\\d\end{bmatrix},
\qquad
\begin{bmatrix}c\\d\end{bmatrix}=P^{-1}X.
$$

So the desired matrix is

$$
A=P^{-1}.
$$

In [ ]:
P = sp.Matrix.hstack(R_exact, B_exact)
A = P.inv()

print("P = [R B] =")
display(P)
print("A = P^{-1} =")
display(A)
print("Decimal A:")
display(sp.N(A, 5))

Thus

$$
A=
\begin{bmatrix}
\frac{400}{37} & -\frac{700}{37}\\[4pt]
-\frac{1650}{37} & \frac{3350}{37}
\end{bmatrix}.
$$

## 5. Task 3: Convert $(c,d,0)$ to the unique coefficients $(r,b,g)$

Since

$$
uR+vB+wG=0,
$$

we may add any multiple $\alpha(uR+vB+wG)$ without changing $X$:

$$
X=(c+\alpha u)R+(d+\alpha v)B+(\alpha w)G.
$$

Define

$$
r=c+\alpha u,\qquad b=d+\alpha v,\qquad g=\alpha w.
$$

Choose $\alpha$ so that

$$
r+b+g=1.
$$

Then

$$
(c+d)+\alpha(u+v+w)=1,
$$

so

$$
\alpha=\frac{1-c-d}{s},
\qquad s=u+v+w.
$$

In [ ]:
def rgb_barycentric_pair(x_value, y_value):
    X = sp.Matrix([sp.Rational(str(x_value)), sp.Rational(str(y_value))])
    c, d = A * X
    alpha = sp.simplify((1 - c - d) / s)
    r = sp.simplify(c + alpha*u)
    b = sp.simplify(d + alpha*v)
    g = sp.simplify(alpha*w)
    return c, d, alpha, r, b, g

c, d, alpha, r, b, g = rgb_barycentric_pair(0.3, 0.5)

print("For X = (0.3, 0.5):")
print("c =", c, "≈", float(c))
print("d =", d, "≈", float(d))
print("alpha =", alpha, "≈", float(alpha))
print("r =", r, "≈", float(r))
print("b =", b, "≈", float(b))
print("g =", g, "≈", float(g))
print("r+b+g =", sp.simplify(r+b+g))

## 6. Task 4: Apply to $X=(0.3,0.5)$

For

$$
X=\begin{bmatrix}0.3\\0.5\end{bmatrix},
$$

the unique coefficients are

$$
r=\frac{51}{226}\approx 0.2257,
\qquad
b=\frac{156}{791}\approx 0.1972,
\qquad
g=\frac{913}{1582}\approx 0.5771.
$$

So

$$
X\approx 0.2257R+0.1972B+0.5771G.
$$

All three coefficients are positive, so this point lies inside the RGB triangle.

In [ ]:
X_np = np.array([0.3, 0.5])
X_reconstructed = float(r)*R + float(b)*B + float(g)*G

print("Original X:", X_np)
print("Reconstructed X:", X_reconstructed)
print("Error:", X_reconstructed - X_np)

plt.figure(figsize=(6,6))
plt.plot(triangle[:,0], triangle[:,1], marker="o")
for name, Pnt in [("R", R), ("B", B), ("G", G)]:
    plt.text(Pnt[0]+0.01, Pnt[1]+0.01, name, fontsize=14)
plt.scatter([X_np[0]], [X_np[1]], s=100)
plt.text(X_np[0]+0.01, X_np[1]+0.01, "X", fontsize=14)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Point X inside the RGB Triangle")
plt.grid(True)
plt.axis("equal")
plt.show()

## 7. Task 5: Complementary color

The complementary color $X'$ is defined by

$$
\frac12X+\frac12X'=W,
$$

where

$$
W=\frac13(R+B+G).
$$

Solving,

$$
X'=2W-X.
$$

In coefficient form,

$$
W=\left(\frac13,\frac13,\frac13\right).
$$

If

$$
X=(r,b,g),
$$

then

$$
X'=
\left(
\frac23-r,\,
\frac23-b,\,
\frac23-g
\right).
$$

In [ ]:
r_comp = sp.simplify(sp.Rational(2,3) - r)
b_comp = sp.simplify(sp.Rational(2,3) - b)
g_comp = sp.simplify(sp.Rational(2,3) - g)

print("Complementary color coefficients:")
print("r' =", r_comp, "≈", float(r_comp))
print("b' =", b_comp, "≈", float(b_comp))
print("g' =", g_comp, "≈", float(g_comp))
print("sum =", sp.simplify(r_comp+b_comp+g_comp))

X_comp = float(r_comp)*R + float(b_comp)*B + float(g_comp)*G
W_np = (R+B+G)/3

print("X' point =", X_comp)
print("Check (X+X')/2 =", (X_np + X_comp)/2)
print("White point W =", W_np)

Therefore the complementary color of $X=(0.3,0.5)$ has coefficients

$$
(r',b',g')
=
\left(
\frac{299}{678},\,
\frac{1114}{2373},\,
\frac{425}{4746}
\right)
\approx
(0.4410,\;0.4694,\;0.0895).
$$

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(triangle[:,0], triangle[:,1], marker="o")
for name, Pnt in [("R", R), ("B", B), ("G", G)]:
    plt.text(Pnt[0]+0.01, Pnt[1]+0.01, name, fontsize=14)

plt.scatter([X_np[0]], [X_np[1]], s=100, label="X")
plt.scatter([X_comp[0]], [X_comp[1]], s=100, label="Complement X'")
plt.scatter([W_np[0]], [W_np[1]], s=100, label="White W")
plt.plot([X_np[0], X_comp[0]], [X_np[1], X_comp[1]], linestyle="--")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Complementary Colors: W is the Midpoint of X and X'")
plt.grid(True)
plt.axis("equal")
plt.legend()
plt.show()

## 8. Summary of key ideas

1. RGB mixtures inside the triangle are affine combinations:
   $$
   rR+bB+gG,\qquad r+b+g=1.
   $$

2. The relation
   $$
   uR+vB+wG=0
   $$
   explains why three planar vectors are linearly dependent.

3. The condition $r+b+g=1$ makes the representation unique.

4. Complementary color is easiest in barycentric coordinates:
   $$
   (r',b',g')=
   \left(\frac23-r,\frac23-b,\frac23-g\right).
   $$